# Cross-Validation Summary & Model Complementarity

이 노트북은 다음을 자동으로 수집·시각화합니다.
- 교차검증 성능/분산: 모델별 Fold F1 평균/표준편차
- 클래스별 취약점: 모델별(평균) per-class accuracy 테이블/히트맵
- 모델 간 보완성: 모델 쌍 간 per-class accuracy 차이(Δ) 시각화 및 요약

실행 전제: 각 run_dir에
- train.log ("Best f1: ...")
- config.yaml (split.fold_index)
- per_class_best.json 또는 per_class_latest.json (per-class 통계)
가 존재해야 합니다.


In [3]:
import json, re, os, glob, time, yaml, math
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8')
ROOT = Path.cwd()
runs = sorted(glob.glob('outputs/runs/*'))
len(runs), runs[:3]

(0, [])

In [4]:
def parse_best_f1(log_path: Path) -> float | None:
    if not log_path.exists():
        return None
    best = None
    for line in log_path.read_text(errors='ignore').splitlines():
        m = re.search(r'Best f1: ([0-9.]+)', line)
        if m:
            best = float(m.group(1))
    return best

def load_fold_index(cfg_path: Path) -> int | None:
    try:
        cfg = yaml.safe_load(cfg_path.read_text())
        return int(cfg['split']['fold_index'])
    except Exception:
        return None

def identify_model(run_name: str) -> str:
    name = run_name.lower()
    if 'tf_efficientnet_b7_ns' in name: return 'effnet_b7'
    if 'swin_large_patch4_window12' in name: return 'swin_large'
    if 'convnextv2' in name: return 'convnextv2'
    if 'efficientnetv2_l' in name: return 'effnetv2_l'
    return 'other'

def load_per_class(run_dir: Path) -> dict | None:
    # prefer best, fallback to latest
    for fname in ['per_class_best.json','per_class_latest.json']:
        p = run_dir / fname
        if p.exists():
            try:
                return json.loads(p.read_text())
            except Exception:
                pass
    return None

records = []
for r in runs:
    rd = Path(r)
    if not rd.is_dir():
        continue
    model = identify_model(rd.name)
    if model == 'other':
        continue
    f1 = parse_best_f1(rd / 'train.log')
    fold = load_fold_index(rd / 'config.yaml')
    pc = load_per_class(rd)
    records.append({'run_dir': rd.name, 'model': model, 'fold': fold, 'best_f1': f1, 'per_class': pc})

df = pd.DataFrame(records).dropna(subset=['fold','best_f1'])
df.sort_values(['model','fold'], inplace=True)
df.head()

KeyError: ['fold', 'best_f1']

## 1) 모델별 교차검증 성능/분산
Fold별 F1로 평균/표준편차를 요약합니다. 막대그래프 + 에러바로 시각화합니다.

In [ ]:
g = df.groupby('model')['best_f1']
summary = g.agg(['mean','std','count']).reset_index()
summary.sort_values('mean', ascending=False, inplace=True)
summary


In [ ]:
plt.figure(figsize=(6,4))
ax = plt.gca()
ax.bar(summary['model'], summary['mean'], yerr=summary['std'], capsize=4, color='#6aaed6', edgecolor='#333')
ax.set_ylabel('CV F1 (mean ± sd)')
ax.set_title('Model-wise CV performance')
plt.tight_layout()
out_dir = Path('reports/summary') / time.strftime('%Y%m%d-%H%M%S')
out_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(out_dir / 'cv_f1_by_model.png', dpi=150); plt.close()
out_dir

## 2) 클래스별 취약점(모델 평균 per-class accuracy)
per_class_best.json의 per_class_accuracy를 fold 평균으로 집계하여 모델별 히트맵을 그립니다.

In [ ]:
# 펼치기: 각 행에 accuracy 벡터를 담은 후 모델별로 평균
rows = []
for _, r in df.iterrows():
    pc = r['per_class'] or {}
    acc = pc.get('per_class_accuracy') or []
    if not acc:
        continue
    rows.append({'model': r['model'], 'fold': r['fold'], 'acc': acc})
pcdf = pd.DataFrame(rows)
pcdf.head()

In [ ]:
# 모델별 평균 per-class accuracy 행렬
mat = {}
for model, gdf in pcdf.groupby('model'):
    arrs = [np.array(v, dtype=float) for v in gdf['acc']]
    # None -> np.nan, 평균 계산시 nanmean 사용
    arrs = [np.array([np.nan if (x is None) else x for x in a]) for a in arrs]
    mean_acc = np.nanmean(np.vstack(arrs), axis=0)
    mat[model] = mean_acc
models = list(mat.keys())
classes = list(range(len(next(iter(mat.values())))))
heat = pd.DataFrame({m: mat[m] for m in models}, index=classes)
plt.figure(figsize=(8,5))
sns.heatmap(heat.T, cmap='Blues', vmin=0, vmax=1)
plt.title('Per-class Accuracy (mean across folds)')
plt.xlabel('Class'); plt.ylabel('Model')
plt.tight_layout(); plt.savefig(out_dir / 'per_class_accuracy_heatmap.png', dpi=150); plt.close()
heat.describe().T[['mean','std']].sort_values('mean', ascending=False)


## 3) 모델 간 보완성 (Δ per-class accuracy)
모델 쌍(A,B)에 대해 per-class accuracy 차이(A-B)를 시각화하여, 
어떤 클래스에서 A가 B보다 강한지(양수), 약한지(음수)를 한눈에 봅니다.
기본 쌍: effnet_b7 vs swin_large


In [ ]:
if 'effnet_b7' in mat and 'swin_large' in mat:
    A = mat['effnet_b7']; B = mat['swin_large']
    delta = A - B
    idx = np.argsort(delta)[::-1]  # 내림차순
    plt.figure(figsize=(10,4))
    xs = np.arange(len(delta))
    plt.bar(xs, delta[idx], color=['#1f78b4' if d>=0 else '#e31a1c' for d in delta[idx]])
    plt.xticks(xs, [str(i) for i in idx], rotation=0)
    plt.axhline(0, color='#333', lw=1)
    plt.ylabel('Δ acc (effnet_b7 - swin_large)')
    plt.title('Complementarity by class (positive: effnet_b7 better)')
    plt.tight_layout(); plt.savefig(out_dir / 'complementarity_effb7_vs_swinl.png', dpi=150); plt.close()
    # 텍스트 요약
    top5 = [(int(i), float(delta[i])) for i in idx[:5]]
    worst5 = [(int(i), float(delta[i])) for i in idx[-5:]]
    print('Top5 classes (effb7 > swinl):', top5)
    print('Worst5 classes (effb7 < swinl):', worst5)
else:
    print('effnet_b7 또는 swin_large 결과가 부족하여 보완성 그래프를 생략합니다.')


## 산출물 경로
- 요약/그림: 위에서 출력한 out_dir 폴더(reports/summary/<timestamp>/)
- 원본 per-class/혼동행렬: 각 run_dir/plots/ 하위 파일 사용 권장

필요 시 모델/쌍을 바꿔 반복 실행하세요.